In [18]:
from pathlib import Path
import shutil

# Archivo original en Google Drive
origen = Path(
    "/content/drive/MyDrive/Colab Notebooks/Simulacion_Rutas_Inundacion.ipynb"
)

# Archivo dentro del repositorio Git
destino = Path(
    "/content/Practica02_Inundacion/Simulacion_Rutas_Inundacion.ipynb"
)

def sincronizar_notebook():
    if not origen.is_file():
        raise FileNotFoundError(
            f"No se encontró el notebook: {origen}"
        )

    if not (destino.parent / ".git").is_dir():
        raise RuntimeError("El repositorio no está clonado")

    shutil.copy2(origen, destino)
    print("Notebook copiado al repositorio correctamente")

In [26]:
from google.colab import userdata

token = userdata.get("GITHUB_TOKEN")

if token:
    print("Token cargado correctamente")
else:
    print("No se encontró el token")

Token cargado correctamente


In [25]:
from google.colab import userdata
from pathlib import Path
import os
import subprocess

# Obtener token guardado en Secrets
token = userdata.get("GITHUB_TOKEN")

if not token:
    raise ValueError("No se encontró GITHUB_TOKEN")

# Configurar un helper de Git que consulta las credenciales
# desde la memoria del proceso, sin escribir el token en disco
import base64

encoded = base64.b64encode(
    f"x-access-token:{token}".encode()
).decode()

# Configuración del repositorio para operaciones HTTPS con GitHub
os.environ["GIT_CONFIG_COUNT"] = "1"
os.environ["GIT_CONFIG_KEY_0"] = "http.https://github.com/.extraheader"
os.environ["GIT_CONFIG_VALUE_0"] = f"AUTHORIZATION: basic {encoded}"

print("Autenticación de Git configurada para esta sesión.")

Autenticación de Git configurada para esta sesión.


# **Crear la matriz de inundación:**

In [36]:
# Dimensiones del terreno
FILAS = 10
COLUMNAS = 10

# Crear una matriz de 10x10 con zonas seguras
terreno = [
    [0 for columna in range(COLUMNAS)]
    for fila in range(FILAS)
]

# Agregar una zona de inundación alta
for columna in range(1, 9):
    terreno[4][columna] = 2

# Agregar una pequeña región bloqueada
for columna in range(3, 7):
    terreno[2][columna] = 3

# Punto inicial y refugio
inicio = (4, 0)
refugio = (4, 9)

# Consultar el terreno como matriz
for fila in terreno:
    print(fila)

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 3, 3, 3, 3, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 2, 2, 2, 2, 2, 2, 2, 2, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


## Definir el costo de cada nivel

In [37]:
# Costos asociados al nivel de inundación
COSTOS = {
    0: 1,
    1: 4,
    2: 9
}

# Nivel que impide el desplazamiento
BLOQUEADO = 3

In [52]:
sincronizar_notebook()

Notebook copiado al repositorio correctamente


In [29]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	new file:   .gitignore
	new file:   Simulacion_Rutas_Inundacion.ipynb
	new file:   evidencias/prueba_1.txt
	new file:   evidencias/prueba_2.txt
	new file:   evidencias/prueba_3.txt
	new file:   evidencias/prueba_4.txt
	new file:   evidencias/prueba_5.txt
	new file:   evidencias/prueba_6.txt



In [28]:
!git add .

In [30]:
!git commit -m "Agregar notebook, estructura de evidencias y creacion de matriz y niveles de inundacion"

[main f35cb34] Agregar notebook, estructura de evidencias y creacion de matriz y niveles de inundacion
 8 files changed, 5 insertions(+)
 create mode 100644 .gitignore
 create mode 100644 Simulacion_Rutas_Inundacion.ipynb
 create mode 100644 evidencias/prueba_1.txt
 create mode 100644 evidencias/prueba_2.txt
 create mode 100644 evidencias/prueba_3.txt
 create mode 100644 evidencias/prueba_4.txt
 create mode 100644 evidencias/prueba_5.txt
 create mode 100644 evidencias/prueba_6.txt


In [31]:
!git push origin main

Enumerating objects: 7, done.
Counting objects: 100% (7/7), done.
Delta compression using up to 2 threads
Compressing objects: 100% (4/4), done.
Writing objects: 100% (6/6), 4.73 KiB | 4.73 MiB/s, done.
Total 6 (delta 0), reused 0 (delta 0), pack-reused 0
To https://github.com/Mald0nad00/Practica02_Inundacion.git
   8310b36..f35cb34  main -> main


## Función para obtener vecinos válidos

In [38]:
def obtener_vecinos(matriz, posicion):
    fila, columna = posicion

    movimientos = [
        (-1, 0),  # Arriba
        (1, 0),   # Abajo
        (0, -1),  # Izquierda
        (0, 1)    # Derecha
    ]

    vecinos = []

    for df, dc in movimientos:
        nueva_fila = fila + df
        nueva_columna = columna + dc

        # Verificar límites del terreno
        if (
            0 <= nueva_fila < len(matriz)
            and 0 <= nueva_columna < len(matriz[0])
        ):

            # Comprobar que la zona sea transitable
            if matriz[nueva_fila][nueva_columna] != BLOQUEADO:
                vecinos.append((nueva_fila, nueva_columna))

    return vecinos

In [39]:
print(obtener_vecinos(terreno, inicio))

[(3, 0), (5, 0), (4, 1)]


## Función para reconstruir el camino

Los algoritmos registrarán desde qué posición llegó el agente a otra posición. Esta función recuperará el recorrido completo.

In [40]:
def reconstruir_camino(anteriores, inicio, refugio):

    if inicio == refugio:
        return [inicio]

    if refugio not in anteriores:
        return None

    camino = []
    actual = refugio

    while actual != inicio:
        camino.append(actual)
        actual = anteriores[actual]

    camino.append(inicio)

    camino.reverse()

    return camino

## Función para verificar inicio y refugio

Esta función comprueba que los datos iniciales sean válidos. Es importante porque el algoritmo no debería intentar buscar rutas sobre un terreno mal definido o con el refugio situado en una región intransitable.

In [41]:
def validar_puntos(matriz, inicio, refugio):

    if not matriz or not matriz[0]:
        raise ValueError("La matriz está vacía")

    columnas = len(matriz[0])

    if any(len(fila) != columnas for fila in matriz):
        raise ValueError("La matriz debe ser rectangular")

    for fila in matriz:
        for nivel in fila:
            if nivel not in COSTOS and nivel != BLOQUEADO:
                raise ValueError("Existe un nivel de inundación inválido")

    for nombre, posicion in [
        ("inicio", inicio),
        ("refugio", refugio)
    ]:
        f, c = posicion

        if not (0 <= f < len(matriz) and 0 <= c < columnas):
            raise ValueError(f"El punto {nombre} está fuera del terreno")

        if matriz[f][c] == BLOQUEADO:
            raise ValueError(f"El punto {nombre} está bloqueado")

## Algoritmo **BFS**

Como todos los movimientos cuentan como un paso, BFS garantiza encontrar un recorrido con la menor cantidad de movimientos cuando existe una ruta. Sin embargo, no garantiza el menor costo de riesgo.

In [51]:
from collections import deque

def bfs(matriz, inicio, refugio):

    validar_puntos(matriz, inicio, refugio)

    # Cola de posiciones pendientes
    cola = deque([inicio])

    # Posiciones descubiertas
    visitados = {inicio}

    # Registro de posiciones anteriores
    anteriores = {}

    # Contador de nodos explorados
    explorados = 0

    while cola:

        actual = cola.popleft()
        explorados += 1

        # Comprobar si llegamos al refugio
        if actual == refugio:
            camino = reconstruir_camino(
                anteriores, inicio, refugio
            )
            return camino, explorados

        # Explorar los vecinos
        for vecino in obtener_vecinos(matriz, actual):

            if vecino not in visitados:

                visitados.add(vecino)

                anteriores[vecino] = actual

                cola.append(vecino)

    # No existe una ruta
    return None, explorados

## Algoritmo Dijkstra

In [ ]:
import heapq

def dijkstra(matriz, inicio, refugio):

    validar_puntos(matriz, inicio, refugio)

    # Cola de prioridad: (costo, posicion)
    cola = [(0, inicio)]

    # Menor costo conocido para cada posición
    distancias = {inicio: 0}

    # Registro del camino
    anteriores = {}

    # Nodos cuyo costo mínimo quedó confirmado
    visitados = set()

    explorados = 0

    while cola:

        costo_actual, actual = heapq.heappop(cola)

        # Evitar procesar posiciones repetidas
        if actual in visitados:
            continue

        visitados.add(actual)
        explorados += 1

        # Comprobar si llegamos al refugio
        if actual == refugio:
            camino = reconstruir_camino(
                anteriores, inicio, refugio
            )
            return camino, explorados

        for vecino in obtener_vecinos(matriz, actual):

            fila, columna = vecino

            # Consultar nivel de inundación
            nivel = matriz[fila][columna]

            # Calcular nuevo costo acumulado
            nuevo_costo = costo_actual + COSTOS[nivel]

            # Revisar si encontramos un camino más barato
            if nuevo_costo < distancias.get(vecino, float("inf")):

                distancias[vecino] = nuevo_costo

                anteriores[vecino] = actual

                heapq.heappush(
                    cola,
                    (nuevo_costo, vecino)
                )

    return None, explorados